# Section 2 – Data Preparation for Electricity Load Forecasting

This notebook covers the full data pipeline:
- Download and load the UCI ElectricityLoadDiagrams20112014 dataset
- Aggregate from 15-minute to hourly resolution
- Filter to 2012-2014 and select clients with complete data
- Handle DST anomalies (23h and 25h days)
- Exploratory Data Analysis (EDA)
- Z-score normalization using only training statistics
- Build calendar covariates
- Create PyTorch Dataset and report window counts
- Leak audit
- Save processed data

## 2.0 – Setup & Imports

In [ ]:
import sys
import os

# Make src/ importable
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'src'))

import urllib.request
import zipfile
import io
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Constants
N_CLIENTS   = 100
TRAIN_END   = '2014-06-30'
VAL_START   = '2014-07-01'
VAL_END     = '2014-09-30'
TEST_START  = '2014-10-01'

DATA_RAW  = os.path.join('..', 'data', 'raw')
DATA_PROC = os.path.join('..', 'data', 'processed')
os.makedirs(DATA_RAW,  exist_ok=True)
os.makedirs(DATA_PROC, exist_ok=True)

print('Setup complete.')

## 2.1 – Download the Dataset

We download the ElectricityLoadDiagrams20112014 dataset from UCI. The file is ~50 MB (zipped).

In [ ]:
URL     = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00321/LD2011_2014.txt.zip'
ZIP_PATH = os.path.join(DATA_RAW, 'LD2011_2014.txt.zip')
TXT_PATH = os.path.join(DATA_RAW, 'LD2011_2014.txt')

if not os.path.exists(TXT_PATH):
    print('Downloading dataset ...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print(f'Downloaded to {ZIP_PATH}')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(DATA_RAW)
    print(f'Extracted to {DATA_RAW}')
else:
    print(f'Dataset already present at {TXT_PATH}')

## 2.2 – Load Raw Data

The dataset contains 15-minute power readings (kW) for 370 Portuguese clients from 2011 to 2014.

In [ ]:
df_raw = pd.read_csv(
    TXT_PATH,
    sep=';',
    decimal=',',
    index_col=0,
    parse_dates=True,
    infer_datetime_format=True
)

# The index name is the datetime; rename for clarity
df_raw.index.name = 'datetime'

print(f'Raw shape: {df_raw.shape}')
print(f'Date range: {df_raw.index.min()} → {df_raw.index.max()}')
print(f'Columns (clients): {df_raw.shape[1]}')
df_raw.head()

## 2.3 – Aggregate 15-min → Hourly (kWh)

Each hourly kWh value is the **sum of 4 consecutive 15-min readings** (4 × kW × 0.25 h = kWh).

In [ ]:
# Resample to hourly by summing 4 quarter-hour slots → gives kWh
df_hourly = df_raw.resample('1H').sum(min_count=1)

# Filter to 2012-2014
df_hourly = df_hourly.loc['2012':'2014']

print(f'Hourly shape after filtering to 2012-2014: {df_hourly.shape}')
print(f'Date range: {df_hourly.index.min()} → {df_hourly.index.max()}')

## 2.4 – Keep Clients with Complete Data

We keep only clients whose NaN rate is below 1% across 2012-2014, then select a random 100-client subset.

In [ ]:
nan_rate = df_hourly.isna().mean()
complete_clients = nan_rate[nan_rate < 0.01].index.tolist()
print(f'Clients with < 1% NaN: {len(complete_clients)}')

# Reproducible 100-client subset
rng = np.random.default_rng(SEED)
chosen = rng.choice(complete_clients, size=min(N_CLIENTS, len(complete_clients)), replace=False)
chosen = sorted(chosen.tolist())

df_panel = df_hourly[chosen].copy()
print(f'Panel shape: {df_panel.shape}  ({df_panel.shape[1]} clients)')

## 2.5 – Handle DST Days (23h and 25h days)

Portugal uses Western European Time (UTC±0 in winter, UTC+1 in summer). On DST transitions:
- **Spring-forward**: one hour is skipped → 23-hour day → one NaN row per client.
- **Fall-back**: one hour appears twice → 25-hour day → duplicate index.

We resolve this by:
1. Reindexing to a continuous UTC hourly grid (no gaps, no duplicates).
2. Interpolating any remaining NaNs linearly (at most 1 h gap per DST event).

In [ ]:
# Build a complete UTC hourly grid covering the full range
full_index = pd.date_range(
    start=df_panel.index.min(),
    end=df_panel.index.max(),
    freq='1H'
)

# Aggregate duplicates (fall-back hours) by averaging before reindex
df_panel = df_panel.groupby(df_panel.index).mean()

# Reindex to full grid (introduces NaN for spring-forward gaps)
df_panel = df_panel.reindex(full_index)

# Linearly interpolate short gaps (limit=2 hours max)
df_panel = df_panel.interpolate(method='time', limit=2)

# Forward-fill any remaining edge NaNs
df_panel = df_panel.ffill().bfill()

print(f'Panel after DST fix: {df_panel.shape}')
print(f'Remaining NaNs: {df_panel.isna().sum().sum()}')

## 2.6 – Exploratory Data Analysis (EDA)

### 2.6.1 Average consumption by hour of day and day of week

In [ ]:
# Mean consumption across all clients
mean_load = df_panel.mean(axis=1).rename('mean_kWh')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Hour-of-day profile
hourly_profile = mean_load.groupby(mean_load.index.hour).mean()
axes[0].bar(hourly_profile.index, hourly_profile.values, color='steelblue')
axes[0].set_xlabel('Hour of day')
axes[0].set_ylabel('Mean kWh')
axes[0].set_title('Average consumption by hour of day')
axes[0].set_xticks(range(0, 24, 2))

# Day-of-week profile
dow_profile = mean_load.groupby(mean_load.index.dayofweek).mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1].bar(days, dow_profile.values, color='coral')
axes[1].set_xlabel('Day of week')
axes[1].set_ylabel('Mean kWh')
axes[1].set_title('Average consumption by day of week')

plt.tight_layout()
plt.show()

### 2.6.2 Portuguese Public Holidays

We mark official Portuguese public holidays and inspect their consumption patterns.

In [ ]:
# Portuguese national holidays (fixed dates)
def portuguese_holidays(years):
    """Return a set of date strings for Portuguese national holidays."""
    fixed = [
        '01-01',  # New Year
        '04-25',  # Freedom Day
        '05-01',  # Labour Day
        '06-10',  # Portugal Day
        '08-15',  # Assumption of Mary
        '10-05',  # Republic Day
        '11-01',  # All Saints
        '12-01',  # Restoration of Independence
        '12-08',  # Immaculate Conception
        '12-25',  # Christmas
    ]
    holidays = set()
    for y in years:
        for d in fixed:
            holidays.add(f'{y}-{d}')
    return holidays

years = [2012, 2013, 2014]
hols  = portuguese_holidays(years)

# Create holiday flag series
date_strs  = df_panel.index.strftime('%Y-%m-%d')
is_holiday = pd.Series(date_strs.isin(hols), index=df_panel.index, name='is_holiday')

print(f'Holiday hours in dataset: {is_holiday.sum()}')

# Compare holiday vs workday profile
hol_profile     = mean_load[is_holiday].groupby(mean_load[is_holiday].index.hour).mean()
workday_profile = mean_load[~is_holiday & (mean_load.index.dayofweek < 5)].groupby(
    mean_load[~is_holiday & (mean_load.index.dayofweek < 5)].index.hour).mean()

plt.figure(figsize=(10, 4))
plt.plot(workday_profile.index, workday_profile.values, label='Workday', color='steelblue')
plt.plot(hol_profile.index,     hol_profile.values,     label='Holiday', color='red', linestyle='--')
plt.xlabel('Hour of day')
plt.ylabel('Mean kWh')
plt.title('Hourly profile: Workday vs Portuguese Holiday')
plt.legend()
plt.xticks(range(0, 24))
plt.tight_layout()
plt.show()

### 2.6.3 Christmas and New Year Consumption

In [ ]:
for year in [2012, 2013]:
    for label, date in [(f'Christmas {year}', f'{year}-12-25'), (f"New Year {year+1}", f'{year+1}-01-01')]:
        try:
            day_data = mean_load.loc[date]
            plt.figure(figsize=(8, 3))
            plt.plot(range(len(day_data)), day_data.values, marker='o', color='darkgreen')
            plt.title(f'Mean load on {label}')
            plt.xlabel('Hour')
            plt.ylabel('Mean kWh')
            plt.xticks(range(0, len(day_data), 2))
            plt.tight_layout()
            plt.show()
        except KeyError:
            print(f'{label} not in dataset range.')

## 2.7 – Scale Analysis Between Clients

In [ ]:
client_avg = df_panel.mean(axis=0).sort_values()

c_min    = client_avg.idxmin()
c_median = client_avg.index[len(client_avg) // 2]
c_max    = client_avg.idxmax()

print('Scale analysis (mean hourly kWh per client):')
print(f'  Min    client: {c_min:10s}  →  {client_avg[c_min]:.3f} kWh')
print(f'  Median client: {c_median:10s}  →  {client_avg[c_median]:.3f} kWh')
print(f'  Max    client: {c_max:10s}  →  {client_avg[c_max]:.3f} kWh')
print(f'  Max/Min ratio: {client_avg[c_max]/client_avg[c_min]:.1f}x')

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, (client, title) in zip(axes, [(c_min, 'Min'), (c_median, 'Median'), (c_max, 'Max')]):
    week_data = df_panel[client].iloc[:24*7]
    ax.plot(week_data.values)
    ax.set_title(f'{title} avg client: {client}\n({client_avg[client]:.2f} kWh/h avg)')
    ax.set_xlabel('Hour (first week of 2012)')
    ax.set_ylabel('kWh')
plt.tight_layout()
plt.show()

## 2.8 – Temporal Split

We perform a **strict temporal split** to avoid data leakage:

| Split | Period |
|-------|--------|
| Train | 2012-01-01 → 2014-06-30 |
| Val   | 2014-07-01 → 2014-09-30 |
| Test  | 2014-10-01 → 2014-12-31 |

In [ ]:
train_df = df_panel.loc[:TRAIN_END]
val_df   = df_panel.loc[VAL_START:VAL_END]
test_df  = df_panel.loc[TEST_START:]

print(f'Train: {train_df.index.min()} → {train_df.index.max()}  ({len(train_df):,} rows)')
print(f'Val:   {val_df.index.min()}   → {val_df.index.max()}   ({len(val_df):,} rows)')
print(f'Test:  {test_df.index.min()}  → {test_df.index.max()}  ({len(test_df):,} rows)')

## 2.9 – Z-score Normalization (Training Stats Only)

We compute mean and standard deviation **per client** using **only training data**, then apply the same statistics to val and test. This prevents leaking future distribution information.

In [ ]:
# Compute normalization statistics from training split ONLY
mean_ = train_df.mean(axis=0)   # shape: (n_clients,)
std_  = train_df.std(axis=0)    # shape: (n_clients,)
std_  = std_.replace(0, 1)      # avoid division by zero for constant series

# Apply to all splits using TRAINING statistics
train_norm = (train_df - mean_) / std_
val_norm   = (val_df   - mean_) / std_
test_norm  = (test_df  - mean_) / std_

print('Z-score normalization applied using training statistics only.')
print(f'Train normalized:  mean={train_norm.values.mean():.4f}, std={train_norm.values.std():.4f}')
print(f'Val   normalized:  mean={val_norm.values.mean():.4f}, std={val_norm.values.std():.4f}')
print(f'Test  normalized:  mean={test_norm.values.mean():.4f}, std={test_norm.values.std():.4f}')

# Store for downstream use
norm_stats = pd.DataFrame({'mean': mean_, 'std': std_})
norm_stats.to_csv(os.path.join(DATA_PROC, 'norm_stats.csv'))
print('\nNormalization stats saved to data/processed/norm_stats.csv')

## 2.10 – Calendar Covariates

We encode time with **sine/cosine features** (to preserve periodicity) and a **holiday flag**:

| Feature | Encoding |
|---------|----------|
| Hour of day | sin/cos with period 24 |
| Day of week | sin/cos with period 7 |
| Month | sin/cos with period 12 |
| Holiday | binary flag (0/1) |

In [ ]:
def make_calendar_features(idx, holidays_set):
    """Build calendar covariate DataFrame for a DatetimeIndex."""
    hour     = idx.hour
    dow      = idx.dayofweek
    month    = idx.month
    date_str = idx.strftime('%Y-%m-%d')

    df_cal = pd.DataFrame(index=idx)
    # Sine/cosine encodings
    df_cal['hour_sin']  = np.sin(2 * np.pi * hour  / 24)
    df_cal['hour_cos']  = np.cos(2 * np.pi * hour  / 24)
    df_cal['dow_sin']   = np.sin(2 * np.pi * dow   / 7)
    df_cal['dow_cos']   = np.cos(2 * np.pi * dow   / 7)
    df_cal['month_sin'] = np.sin(2 * np.pi * month / 12)
    df_cal['month_cos'] = np.cos(2 * np.pi * month / 12)
    # Holiday flag (future-known covariate — safe to use)
    df_cal['is_holiday'] = date_str.isin(holidays_set).astype(np.float32)
    return df_cal

cal_train = make_calendar_features(train_df.index, hols)
cal_val   = make_calendar_features(val_df.index,   hols)
cal_test  = make_calendar_features(test_df.index,  hols)

print(f'Calendar features shape (train): {cal_train.shape}')
print('Features:', cal_train.columns.tolist())
cal_train.head()

## 2.11 – PyTorch Dataset

We import the `ElectricityDataset` class from `src/dataset.py`.

In [ ]:
try:
    from dataset import ElectricityDataset
    print('ElectricityDataset imported from src/dataset.py')
except ImportError:
    print('src/dataset.py not found — defining a minimal version for demonstration.')
    import torch
    from torch.utils.data import Dataset

    class ElectricityDataset(Dataset):
        """
        Sliding-window dataset for multivariate electricity load forecasting.

        Parameters
        ----------
        values : np.ndarray, shape (T, N)
            Normalized load series (T timesteps, N clients).
        calendar : np.ndarray, shape (T, C)
            Calendar covariates aligned to `values`.
        lookback : int
            Number of past hours fed as input (L).
        horizon : int
            Number of future hours to predict (H).
        stride : int
            Step between successive windows.
        """
        def __init__(self, values, calendar, lookback=168, horizon=24, stride=1):
            super().__init__()
            self.values   = torch.tensor(values,   dtype=torch.float32)
            self.calendar = torch.tensor(calendar, dtype=torch.float32)
            self.L  = lookback
            self.H  = horizon
            self.S  = stride
            self.n_windows = (len(values) - lookback - horizon) // stride + 1

        def __len__(self):
            return self.n_windows

        def __getitem__(self, idx):
            t = idx * self.S
            x_load = self.values[t : t + self.L]              # (L, N)
            x_cal  = self.calendar[t : t + self.L]            # (L, C)
            y      = self.values[t + self.L : t + self.L + self.H]  # (H, N)
            cal_fut= self.calendar[t + self.L : t + self.L + self.H]  # (H, C)
            return x_load, x_cal, y, cal_fut

print('ElectricityDataset ready.')

## 2.12 – Window Report

For **L=168 (1 week lookback)**, **H=24 (1 day horizon)**, **stride=1**, we report the number of windows per split.

In [ ]:
L, H, STRIDE = 168, 24, 1

def count_windows(n_timesteps, lookback, horizon, stride):
    return max(0, (n_timesteps - lookback - horizon) // stride + 1)

n_train = count_windows(len(train_norm), L, H, STRIDE)
n_val   = count_windows(len(val_norm),   L, H, STRIDE)
n_test  = count_windows(len(test_norm),  L, H, STRIDE)

print(f'Window parameters: L={L}, H={H}, stride={STRIDE}')
print(f'  Train timesteps: {len(train_norm):,}  →  {n_train:,} windows')
print(f'  Val   timesteps: {len(val_norm):,}    →  {n_val:,} windows')
print(f'  Test  timesteps: {len(test_norm):,}   →  {n_test:,} windows')
print(f'  Total windows  : {n_train + n_val + n_test:,}')

# Instantiate datasets as a sanity check
try:
    ds_train = ElectricityDataset(train_norm.values, cal_train.values, L, H, STRIDE)
    ds_val   = ElectricityDataset(val_norm.values,   cal_val.values,   L, H, STRIDE)
    ds_test  = ElectricityDataset(test_norm.values,  cal_test.values,  L, H, STRIDE)
    print(f'\nDataset lengths: train={len(ds_train)}, val={len(ds_val)}, test={len(ds_test)}')
    x_load, x_cal, y, cal_fut = ds_train[0]
    print(f'Sample shapes → x_load: {tuple(x_load.shape)}, x_cal: {tuple(x_cal.shape)}, '
          f'y: {tuple(y.shape)}, cal_fut: {tuple(cal_fut.shape)}')
except Exception as e:
    print(f'Dataset instantiation skipped (torch not available): {e}')

## 2.13 – Leak Audit

This section explicitly demonstrates the **4 potential data leaks** and shows how our pipeline avoids each one.

---

### Leak 1 – Random Window Splits
**Risk**: Randomly shuffling windows can place future context in the training set.  
**Mitigation**: We use **strict temporal splits** on raw timesteps before creating any windows.

In [ ]:
# PROOF: all training windows end before VAL_START
last_train_window_end = train_norm.index[L + H - 1]  # earliest possible last step
first_val_start       = val_norm.index[0]

assert last_train_window_end < first_val_start, 'LEAK: training window overlaps validation!'
print(f'[PASS] Last index reachable by first train window: {last_train_window_end}')
print(f'       First validation index                    : {first_val_start}')
print('       No overlap → Leak 1 AVOIDED.')

### Leak 2 – Scaling with Full Series
**Risk**: Computing mean/std over the entire series exposes future statistics to the model during training.  
**Mitigation**: `mean_` and `std_` are computed **only on `train_df`** and then re-applied to val/test.

In [ ]:
# PROOF: normalization statistics were derived ONLY from training data
# Simulate a leaky scaler that uses the full panel
leaky_mean = df_panel.mean(axis=0)
leaky_std  = df_panel.std(axis=0)

max_mean_diff = (mean_ - leaky_mean).abs().max()
max_std_diff  = (std_  - leaky_std).abs().max()

print(f'Max |train_mean - full_mean| across clients : {max_mean_diff:.4f}')
print(f'Max |train_std  - full_std|  across clients : {max_std_diff:.4f}')
print('Statistics differ → using full-series stats WOULD shift the normalization.')
print('[PASS] We use only training mean_/std_ → Leak 2 AVOIDED.')

### Leak 3 – Early Stopping with Test Set
**Risk**: Using test-set loss to decide when to stop training exposes the model to test information.  
**Mitigation**: Early stopping is performed exclusively on **validation loss**. Test is never observed during training.

In [ ]:
# Pseudo-code pattern showing correct early-stopping setup
early_stop_pseudocode = '''
best_val_loss = float('inf')
patience_counter = 0
PATIENCE = 5

for epoch in range(MAX_EPOCHS):
    train_one_epoch(model, train_loader)          # uses ds_train only
    val_loss = evaluate(model, val_loader)        # uses ds_val only
    # ← test_loader is NEVER touched here

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stop at epoch {epoch}")
            break

# Evaluate on test ONLY after training is complete
load_checkpoint(model)
test_loss = evaluate(model, test_loader)  # final, one-time evaluation
'''
print('Correct early-stopping pattern:')
print(early_stop_pseudocode)
print('[PASS] Early stopping uses val_loader only → Leak 3 AVOIDED.')

### Leak 4 – Misaligned Covariates
**Risk**: Including covariates that depend on future target values (e.g., future load aggregates) as model inputs.  
**Mitigation**: Our calendar covariates (hour, weekday, month, holiday) are **fully determined by the calendar** — they are known in advance without observing future load.

In [ ]:
# PROOF: calendar features are derivable from timestamps alone
# They do not depend on y (future load values)
print('Calendar covariate columns:', cal_train.columns.tolist())
print()
print('Checking that covariates are independent of load values...')

# Verify no feature correlates perfectly with target (they are purely time-based)
sample_client = df_panel.columns[0]
for feat in cal_train.columns:
    corr = cal_train[feat].corr(train_df[sample_client])
    print(f'  Pearson r({feat:12s}, load_{sample_client}) = {corr:+.3f}')

print()
print('None of the calendar features IS the target — they are future-known time markers.')
print('[PASS] All covariates are future-known calendar features → Leak 4 AVOIDED.')

## 2.14 – Save Processed Panel

We save the (unnormalized) hourly panel to `data/processed/panel_hourly.parquet` for reproducibility. The normalization statistics are already saved as `norm_stats.csv`.

In [ ]:
parquet_path = os.path.join(DATA_PROC, 'panel_hourly.parquet')
df_panel.to_parquet(parquet_path, compression='snappy')

print(f'Saved panel_hourly.parquet → {parquet_path}')
print(f'  Shape : {df_panel.shape}')
print(f'  Size  : {os.path.getsize(parquet_path) / 1e6:.2f} MB')

# Quick read-back verification
verify = pd.read_parquet(parquet_path)
assert verify.shape == df_panel.shape, 'Shape mismatch on read-back!'
print('Read-back verification: OK')

## Summary

| Step | Result |
|------|--------|
| Raw data | 370 clients, 15-min resolution, 2011-2014 |
| Hourly aggregation | Sum 4×15-min → kWh |
| Filtered period | 2012-2014 |
| Selected clients | 100 (SEED=42) |
| DST fix | Reindex + time interpolation |
| Normalization | Z-score per client, train stats only |
| Calendar features | 7 features (sin/cos × 3 + holiday) |
| Train windows (L=168, H=24) | computed above |
| Leaks audited | 4 / 4 AVOIDED |
| Output file | `data/processed/panel_hourly.parquet` |